# Figure 5 — the connectional reorganisation of human association cortex

Where the mouse cannot rebuild human connectivity is not noise. It is a specific, network-shaped
territory that tracks cortical expansion. This notebook derives that from scratch.

Reconstruction-coverage asks, for each human parcel, how well its functional-connectivity
fingerprint is rebuilt by routing mouse connectivity through π:

```
pihat = pi / pi.sum(0)          # column-normalised: each human parcel's mouse mixture
pred  = pihat.T @ Mfc @ pihat   # the human FC matrix the mouse implies
coverage[j] = pearson(pred[j], Hfc[j])   # diagonal excluded
```

This replaced an earlier measure that did not work. That measure was the total mouse mass a
parcel received, `log10(pi.sum(0))`, which is dominated by how spatially isolated a parcel is
rather than by whether the mouse can account for it, and whose tail is set by the entropic
regularisation. Reconstruction-coverage is position-decoupled and asks a functional question.

A note on where the numbers come from. The published maps are external data and live in
`data_external/published_cortical_maps.json`. Every statistic below is recomputed here. Older
per-map statistics are still stored in `outputs/logs/section5_evolution_battery.json`, but that
file carries no coupling provenance, so which π produced them cannot be established. They are not
read.

Before running: `python scripts/fetch_data.py`. No results files are needed.

In [ ]:
import json, subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGS = ROOT.parent / 'manuscript' / 'figures'
sys.path.insert(0, str(ROOT / 'src'))

from homer.data import load_cached, load_pi, pi_provenance
from homer.eval.nulls import _haar_rotation

pi = load_pi()
PROV = pi_provenance()
M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))
Mfc = np.asarray(M.uns['fc_mean'], float)
Hfc = np.asarray(H.uns['fc_mean'], float)
xyz = H.var[['x', 'y', 'z']].to_numpy(float)
print(f"coupling {pi.shape[0]:,} x {pi.shape[1]:,}   {PROV['pi_file']}   sha {PROV['pi_sha256'][:16]}...")

PUBLISHED = {
    'visual parcel r':          (0.77, 0.02),
    'dlPFC parcel r':           (0.07, 0.02),
    'association tertile mean': (0.40, 0.01),
    'sensorimotor tertile mean':(0.50, 0.01),
    'tertile gap':              (0.10, 0.01),
    "Cohen's d":                (0.81, 0.03),
    'tertile spin p':           (0.010, 0.010),
    'ContB gap, whole cortex':  (-0.63, 0.03),   # see the reconciliation in section 4
}

def check(name, value):
    exp, tol = PUBLISHED[name]
    ok = abs(value - exp) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:28s} computed {value:+.4g}   manuscript {exp}")
    return ok

## 1. Computing reconstruction-coverage (Fig. 5a, 5b)

Three lines of linear algebra and one correlation per parcel. The diagonal is excluded so that a
parcel's self-connectivity cannot inflate its own score.

In [ ]:
ph = pi.sum(0)
pihat = pi / np.maximum(ph, 1e-300)
pred = pihat.T @ Mfc @ pihat

n = pred.shape[0]
rc = np.full(n, np.nan)
for j in range(n):
    a, b = pred[j].copy(), Hfc[j].copy()
    a[j] = np.nan; b[j] = np.nan                 # exclude the diagonal
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() > 10 and a[ok].std() > 1e-9:
        rc[j] = np.corrcoef(a[ok], b[ok])[0, 1]

print(f"reconstruction-coverage over {np.isfinite(rc).sum():,} of {n:,} human parcels")
print(f"  mean {np.nanmean(rc):.3f}   median {np.nanmedian(rc):.3f}   "
      f"range {np.nanmin(rc):.2f} to {np.nanmax(rc):.2f}")

# It is position-decoupled by construction. The retired mass metric was not: check both against
# spatial isolation, the confound that sank it.
iso = np.load(ROOT / 'outputs/anndata/full_costs.npz')['M_xyz'].min(0)
mass = np.log10(np.maximum(ph, 1e-300))
f = np.isfinite(rc) & np.isfinite(iso) & np.isfinite(mass)
print(f"\ncorrelation with spatial isolation (the confound that retired the mass metric):")
print(f"  reconstruction-coverage  rho = {spearmanr(rc[f], iso[f]).statistic:+.3f}")
print(f"  retired mass-coverage    rho = {spearmanr(mass[f], iso[f]).statistic:+.3f}")

## 2. Two worked parcels (Fig. 5a)

The metric is a slope. One central-visual parcel is rebuilt faithfully; one dorsolateral
prefrontal parcel is not.

In [ ]:
rows = [l.split('\t') for l in (ROOT / 'outputs/anndata/_schaefer_order.txt').read_text().splitlines() if l.strip()]
nr = np.asarray(json.loads((ROOT / 'data_external/human_sc_meta.json').read_text())['node_region'], int)
name_of = {int(p[0]): p[1] for p in rows}
net = np.array([name_of.get(int(k), '?').split('_', 2)[2].split('_')[0]
                if name_of.get(int(k)) else '?' for k in nr])

def best(substr):
    '''Highest-coverage parcel whose Schaefer label contains substr (and lowest, for contrast).'''
    idx = [j for j in range(n) if np.isfinite(rc[j]) and substr in name_of.get(int(nr[j]), '')]
    return max(idx, key=lambda j: rc[j]), min(idx, key=lambda j: rc[j])

vis, _ = best('VisCent')
_, dlp = best('ContB')
print(f"central visual parcel : r = {rc[vis]:+.3f}")
print(f"dlPFC parcel          : r = {rc[dlp]:+.3f}\n")
check('visual parcel r', rc[vis])
check('dlPFC parcel r', rc[dlp])

## 3. Does coverage track cortical expansion? (Fig. 5c, 5d)

Reconstruction-coverage is correlated against published cortical maps, each tested with a spin
null that preserves spatial autocorrelation. The map values are external data; the statistics are
computed here.

Negative ρ means coverage is lower toward the expanded, association end, so the mouse accounts
for less of the cortex that grew most.

In [ ]:
# Published cortical maps: EXTERNAL DATA, so they live in data_external/ rather than in
# outputs/logs/. Only the map values are read; every statistic below is recomputed.
batt = json.loads((ROOT / 'data_external/published_cortical_maps.json').read_text())['maps']

def region_series(mapkey):
    '''Region-mean coverage and map value over the Schaefer regions the map defines.

    Schema mirrors make_fig5_panels.py exactly: each entry carries parallel 'schaefer_ids' and
    'map_values' arrays. Note the entry ALSO carries 'spearman' and 'spin_p' -- those are the
    retired mass-coverage statistics and are deliberately not read here.
    '''
    v = batt[mapkey]
    mp = dict(zip(np.asarray(v['schaefer_ids'], int), np.asarray(v['map_values'], float)))
    ids = [k for k in range(1, 401) if (nr == k).any() and k in mp]
    cc = np.array([np.nanmean(rc[nr == k]) for k in ids])
    mv = np.array([mp[k] for k in ids])
    cen = np.array([xyz[nr == k].mean(0) for k in ids])
    return ids, cc, mv, cen

def spin(cc, mv, cen, N=1000, seed=0):
    '''Spatial spin null: rotate the region centroids, re-match by nearest neighbour.'''
    rng = np.random.default_rng(seed)
    s = cen - cen.mean(0); t = cKDTree(s)
    perms = [t.query(s @ _haar_rotation(rng).T)[1] for _ in range(N)]
    rho = spearmanr(cc, mv).statistic
    null = np.abs([spearmanr(cc[p], mv).statistic for p in perms])
    return rho, (np.sum(null >= abs(rho)) + 1) / (N + 1)

MAPS = [('Sydnor2021 S–A axis', 'hierarchy'), ('Margulies2016 principal gradient', 'hierarchy'),
        ('HCP T1w/T2w hierarchy', 'hierarchy'), ('Xu2020 mouse–human FC homology', 'evolution'),
        ('Hill2010 developmental expansion', 'evolution'),
        ('Xu2020 mouse→human expansion', 'evolution'),
        ('Hill2010 macaque→human expansion', 'evolution')]

print(f"{'map':44s} {'rho':>7s} {'spin p':>8s}")
results = {}
for key, grp in MAPS:
    ids, cc, mv, cen = region_series(key)
    rho, p = spin(cc, mv, cen)
    results[key] = (rho, p)
    print(f"  {key:42s} {rho:+.3f} {p:8.3f}{'  *' if p < 0.05 else ''}")
print(f"\n{sum(1 for r, p in results.values() if p < 0.05)} of {len(results)} clear the spin null")


## 4. How large is the deficit, and what shape is it? (Fig. 5e, 5f)

Split cortex by the sensorimotor–association axis and compare tertiles, then ask which Yeo-17
network carries the deficit. The distributions overlap; what separates them is the floor.

In [ ]:
ids, cc, sa, cen = region_series('Sydnor2021 S–A axis')
order = np.argsort(sa)
k = len(order) // 3
assoc, senso = cc[order[-k:]], cc[order[:k]]
gap = senso.mean() - assoc.mean()
d = gap / np.sqrt((assoc.var(ddof=1) + senso.var(ddof=1)) / 2)

# tertile contrast under the same spin null, so the split is not free
rng = np.random.default_rng(0); s = cen - cen.mean(0); t = cKDTree(s)
null = []
for _ in range(1000):
    p_ = t.query(s @ _haar_rotation(rng).T)[1]
    c = cc[p_]
    null.append(abs(c[order[:k]].mean() - c[order[-k:]].mean()))
p_spin = (np.sum(np.array(null) >= abs(gap)) + 1) / 1001

print(f"association tertile  mean {assoc.mean():.3f}   floor {assoc.min():.2f}")
print(f"sensorimotor tertile mean {senso.mean():.3f}   floor {senso.min():.2f}")
print(f"gap {gap:.3f}   Cohen's d {d:.2f}   spin p {p_spin:.3f}\n")
check('association tertile mean', assoc.mean())
check('sensorimotor tertile mean', senso.mean())
check('tertile gap', gap)
check("Cohen's d", d)
check('tertile spin p', p_spin)

In [ ]:
# Which network carries it? Coverage z-scored across cortex, then each network compared with
# THE REST OF CORTEX -- a gap, not a mean.
#
# THE STATISTIC MATTERS. The caption's -0.69 SD is the gap (ContB mean minus non-ContB mean in SD
# units), which is what the spin null in 11_dlpfc_deficit.py tests. The plain within-network
# z-mean is -0.60. Both are defensible descriptions; only one is the published number, and
# quoting the other would look like a discrepancy in a figure that has none.
m = np.isfinite(rc)
z = (rc[m] - np.nanmean(rc[m])) / np.nanstd(rc[m])
netm = net[m]
nets = sorted({u for u in set(netm) if (netm == u).sum() >= 10})

def gap_sd(u):
    return float(z[netm == u].mean() - z[netm != u].mean())

nm = {u: gap_sd(u) for u in nets}
zmean = {u: float(z[netm == u].mean()) for u in nets}

print('reconstruction-coverage by network (gap from the rest of cortex, SD):')
for u in sorted(nm, key=nm.get):
    lab = 'Control B (dlPFC)' if u == 'ContB' else u
    print(f"  {lab:22s} gap {nm[u]:+.2f}   within-network mean {zmean[u]:+.2f}")
print()
check('ContB gap, whole cortex', nm['ContB'])

# RECONCILIATION, computed rather than read.
#
# The caption quotes -0.69 SD (spin p = 0.006), which is the same statistic on a RESTRICTED
# parcel set: the molecular control z-scores over parcels where reconstruction-coverage AND
# transcriptomic similarity are both defined, so the two can be compared like for like. Whole
# cortex gives -0.63. Neither is wrong; they answer slightly different questions, and the paper
# reports the one that supports the molecular comparison.
#
# This used to read section5_dlpfc_deficit.json. It is a spin null on coverage, which is cheap,
# so reading it from a log was convenience rather than necessity -- the same shortcut this
# notebook exists to avoid.
D = ROOT / 'data_external'
mg, hg = np.load(D / 'mouse_genes_aligned.npy'), np.load(D / 'human_genes_aligned.npy')

def _zc(A):
    A = A.astype(float)
    mu, sd = np.nanmean(A, 0), np.nanstd(A, 0)
    sd[sd < 1e-9] = 1.0
    return (A - mu) / sd

# transcriptomic similarity: for each human parcel, its best correlation to any mouse parcel
Mz, Hz = _zc(mg), _zc(hg)
mok, hok = np.isfinite(Mz).all(1), np.isfinite(Hz).all(1)
Mc = Mz[mok] - Mz[mok].mean(1, keepdims=True)
Mn = Mc / np.linalg.norm(Mc, axis=1, keepdims=True)
Hc = Hz - np.nanmean(Hz, 1, keepdims=True)
Hn = np.full_like(Hc, np.nan)
Hn[hok] = Hc[hok] / np.linalg.norm(Hc[hok], axis=1, keepdims=True)
with np.errstate(invalid='ignore'):
    mol = np.nanmax(Hn @ Mn.T, axis=1)

# the molecular-control mask: both measures defined
mr = np.isfinite(rc) & np.isfinite(mol)
zr_rec = (rc[mr] - rc[mr].mean()) / rc[mr].std()
zr_mol = (mol[mr] - mol[mr].mean()) / mol[mr].std()
sel_r = net[mr] == 'ContB'

def block_gap(sig, sel, coords, n=1000, seed=0):
    """(mean over sel) - (mean over the rest), with a spatial spin null."""
    c = coords - coords.mean(0)
    sph = c / np.linalg.norm(c, axis=1, keepdims=True)
    tree = cKDTree(sph)
    rng = np.random.default_rng(seed)
    perms = [tree.query(sph @ _haar_rotation(rng).T)[1] for _ in range(n)]
    f = lambda s: s[sel].mean() - s[~sel].mean()
    obs = f(sig)
    null = np.abs([f(sig[p]) for p in perms])
    return float(obs), float((np.sum(null >= abs(obs)) + 1) / (n + 1))

g_rec, p_rec = block_gap(zr_rec, sel_r, xyz[mr])
g_mol, p_mol = block_gap(zr_mol, sel_r, xyz[mr])

print(f"\nmolecular control (n = {int(mr.sum()):,} parcels where both are defined):")
print(f"  reconstruction-coverage   {g_rec:+.2f} SD  spin p = {p_rec:.3f}")
print(f"  transcriptomic similarity {g_mol:+.2f} SD  spin p = {p_mol:.3f}")
print(f"  whole-cortex gap computed above: {nm['ContB']:+.2f} SD")
print("\ndlPFC is molecularly mouse-like and connectionally unreachable. That dissociation,")
print("rather than the size of either number, is the result.")

## 5. Build the figure panels

In [ ]:
RUN_PANELS = True

if RUN_PANELS:
    r = subprocess.run([sys.executable, str(FIGS / 'fig5' / 'make_fig5_panels.py')],
                       cwd=str(ROOT), capture_output=True, text=True)
    print((r.stdout or r.stderr)[-600:])
    print('ok' if r.returncode == 0 else f'FAILED ({r.returncode})')
else:
    print('skipped; set RUN_PANELS = True')

## 6. Where §5 stands

The mouse rebuilds sensorimotor, auditory and visual connectivity well, and prefrontal and
lateral temporal connectivity poorly. That deficit tracks evolutionary expansion, concentrates in
one network, and is not explained by spatial position or by transcriptomic dissimilarity. It
turns "the mouse is a poor model of association cortex" from an assumption into a measurement.

In [ ]:
print(f"coupling                {PROV['pi_file']}")
print(f"coverage                mean {np.nanmean(rc):.2f} over {np.isfinite(rc).sum():,} parcels")
print(f"worked parcels          visual {rc[vis]:.2f}   dlPFC {rc[dlp]:.2f}")
print(f"tertile contrast        {assoc.mean():.2f} vs {senso.mean():.2f}  (d = {d:.2f}, spin p = {p_spin:.3f})")
print(f"network extreme         ContB {nm['ContB']:+.2f} SD whole cortex "
      f"({g_rec:+.2f} on the molecular-control set)")
print()
ok = all([check('visual parcel r', rc[vis]),
          check('dlPFC parcel r', rc[dlp]),
          check('association tertile mean', assoc.mean()),
          check('sensorimotor tertile mean', senso.mean()),
          check('tertile gap', gap),
          check("Cohen's d", d),
          check('tertile spin p', p_spin),
          check('ContB gap, whole cortex', nm['ContB'])])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED -- text and code have diverged')
